In [ ]:
import scanpy as sc
import scanpy.external as sce
import pandas as pd
import numpy as np
import os
import shutil
import triku as tk
import matplotlib.pyplot as plt
import matplotlib as mpl
import subprocess
from scipy.sparse import csr_matrix
from IPython.display import display, HTML
import mygene as mg
import seaborn as sns
from tqdm import tqdm
# from tqdm.notebook import tqdm

from bokeh.io import show, output_notebook, reset_output

from scipy.sparse import csr_matrix, csc_matrix

reset_output()
output_notebook()

In [ ]:
from pyfuncs.general import preprocessing_adata_sub, plot_volcano


In [ ]:
from cellassign import assign_cats

In [ ]:
magma = [plt.get_cmap('magma')(i) for i in np.linspace(0,1, 80)]
magma[0] = (0.88, 0.88, 0.88, 1)
magma = mpl.colors.LinearSegmentedColormap.from_list("", magma[:65])

seed = 0

In [ ]:
from matplotlib import font_manager, rcParams

sns.set_style("white")

font_path = "src/fonts/texgyreheros-regular.otf" # Alternative
font_path = "/usr/share/texmf/fonts/opentype/public/tex-gyre/texgyreheros-regular.otf"

mpl.rcParams.update({'font.size': 22})


# Path to the Helvetica font file
custom_font = font_manager.FontProperties(fname=font_path)

# Add the font to Matplotlib's font manager
font_manager.fontManager.addfont(font_path)

# Set the font globally
rcParams['font.family'] = custom_font.get_name()
rcParams['font.size'] = 16
rcParams['figure.dpi'] = 300

## Oprescu adata load

In [ ]:
data_dir = 'data/'
oprescu_dir = data_dir + '/oprescu'

In [ ]:
adata_oprescu = sc.read_loom(oprescu_dir + '/adata_oprescu.loom')

In [ ]:
adata_oprescu.obs_names = adata_oprescu.obs['obs_names']
adata_oprescu.var_names = adata_oprescu.var['var_names']

In [ ]:
adata_oprescu.obs['batch'] = [i.split('_')[0] for i in adata_oprescu.obs_names]

In [ ]:
adata_oprescu.obs

In [ ]:
# Basic QC filtering
adata_oprescu.var['mt'] = adata_oprescu.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_oprescu, qc_vars=['mt'], percent_top=None, inplace=True)

In [ ]:
sc.pl.violin(adata_oprescu, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

sc.pl.scatter(adata_oprescu, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(adata_oprescu, x='total_counts', y='n_genes_by_counts', color='batch')

In [ ]:
sc.pp.filter_cells(adata_oprescu, min_genes=150)

In [ ]:
adata_oprescu_d0 = adata_oprescu[adata_oprescu.obs['batch'] == 'Noninjured'].copy()
adata_oprescu_d05 = adata_oprescu[adata_oprescu.obs['batch'] == 'X0.5.DPI'].copy()
adata_oprescu_d2 = adata_oprescu[adata_oprescu.obs['batch'] == 'X2.DPI'].copy()
adata_oprescu_d35 = adata_oprescu[adata_oprescu.obs['batch'] == 'X3.5.DPI'].copy()
adata_oprescu_d5 = adata_oprescu[adata_oprescu.obs['batch'] == 'X5.DPI'].copy()
adata_oprescu_d10 = adata_oprescu[adata_oprescu.obs['batch'] == 'X10.DPI'].copy()
adata_oprescu_d21 = adata_oprescu[adata_oprescu.obs['batch'] == 'X21.DPI'].copy()

In [ ]:
adata_oprescu_d0

In [ ]:
for adata_oprescu in [adata_oprescu_d0, adata_oprescu_d05, adata_oprescu_d2, adata_oprescu_d35, adata_oprescu_d5, adata_oprescu_d10, adata_oprescu_d21]:
    print(adata_oprescu_d0)
    sc.pp.filter_genes(adata_oprescu, min_counts=1)
    sc.pp.normalize_per_cell(adata_oprescu)
    sc.pp.log1p(adata_oprescu)
    
    sc.pp.pca(adata_oprescu, random_state=seed, n_comps=30)
    sc.pp.neighbors(adata_oprescu, random_state=seed, n_neighbors=int(len(adata_oprescu) ** 0.5 // 2), metric='cosine')
    tk.tl.triku(adata_oprescu)
    
    sc.tl.umap(adata_oprescu, min_dist=0.1, random_state=seed)
    sc.tl.leiden(adata_oprescu, resolution=3, random_state=seed)
    sc.pl.umap(adata_oprescu, color=['leiden', 'batch', 'n_counts'], legend_loc='on data')

In [ ]:
dict_cats_general = {'Lum+ FAP': ['Apod', 'Lum', 'Ly6a', 'Pdgfra', 'Mfap5', 'Dcn'], 
                     'Prg4+ FAP': ['Prg4', 'Fbn1', 'Ly6a', 'Pdgfra', 'Mfap5', 'Dcn'], 
                     'Endothelial': ['Pecam1', 'Kdr', 'Fabp4', 'Cav1', 'Cdh5', 'Tek'], 
                     'Pericyte': ['Rgs5', 'Notch3', 'Myl9', 'Ndufa4l2', 'Itga7', 'Myh11', 'Pln', 'Abcc9'], 
                     'Satellite cell': ['Pax7', 'Myod1', 'Chodl', 'Vcam1', 'Sdc4', 'Myf5',], 
                     'Myonuclei': ['Tnnc2', 'Myh4', 'Acta1', 'Ckm', 'Tpm2', 'Eno3', 'Slc25a4'], 
                     'Tenocyte': ['Scx', 'Tnmd', 'Mkx', 'Col12a1', 'Col1a1', 'Tnc', 'Fmod', 'Comp'], 
                     'Neural cell': ['Mpz', 'Ptn', 'S100b'], 
                     'Glial cell': ['Plp1', 'Kcna1', 'S100b', 'Mbp', 'Mpz',],
                     'Guide cell': ['Ncam2'],
                     'Immune': ['H2-Aa', 'Cd74'], 
                     'APC': ['H2-Eb1', 'H2-Ab1'],
                     'APC / Proliferative ICs': ['Mki67', 'Top2a'], 
                     'B cell': ['Cd19', 'Cd22', 'Ms4a1', 'Ptprc'], 
                     'T cell': ['Cd3d', 'Cd3e', 'Cd3g', 'Cd8a', 'Cd4', 'Ptprc', 'Cd28'], 
                     'Monocyte': ['Csf1r', 'Adgre1'], 
                     'Macrophage': ['Itgam', 'Csf1r', 'Adgre1', 'Itgb1', 'Cd68'],
                     'Myeloid': ['Clec12a', 'Acp5'], 
                     'Neutrophil': ['S100a8', 'S100a9', 'Itgam', 'Cd14', ], 
                     'Epcam+': ['Epcam']}

In [ ]:
A_markers = ['6030408B16Rik', 'Col9a2', 'Dlk1', 'Shisa3',  'Saa1',  'Nipal1']
A_markers_extra = ['Kcnk2',  # Not specific enough
                   'Adamtsl2',  # Not specific enough
                   'Cst6',  # Teno marker
                   'Sorcs2',  # Not specific enough
                   'Susd5',  # Not specific enough
                   'Rgs17',  # Not specific enough
                   'Gfra2']  # Marks immune population
B_markers = ['Lypd2', 'Wnt6', 'Cldn1', 'Moxd1', 'Mansc4', 'Dleu7', 'Efnb3', 'Stra6', 'Sbspon', 'Ace2', 'Hcn4', 'Cldn22', 'Wnt10a', 'Ocln']  

### A_markers

In [ ]:
sc.pl.umap(adata_oprescu_d0, color=['leiden', 'Tnc', 'Tnmd', 'Pdgfra', 'Lum', 'Prg4', 'Pdpn'] + [i for i in A_markers if i in adata_oprescu_d0.var_names], legend_loc='on data', ncols=3, cmap=magma)

In [ ]:
sc.pl.umap(adata_oprescu_d05, color=['leiden', 'Tnc', 'Tnmd', 'Pdgfra', 'Lum', 'Prg4', 'Pdpn'] + [i for i in A_markers if i in adata_oprescu_d05.var_names], legend_loc='on data', ncols=3, cmap=magma)

In [ ]:
sc.pl.umap(adata_oprescu_d2, color=['leiden', 'Tnc', 'Tnmd', 'Pdgfra', 'Lum', 'Prg4', 'Pdpn'] + [i for i in A_markers if i in adata_oprescu_d2.var_names], legend_loc='on data', ncols=3, cmap=magma)

In [ ]:
sc.pl.umap(adata_oprescu_d35, color=['leiden', 'Tnc', 'Tnmd', 'Pdgfra', 'Lum', 'Prg4', 'Pdpn'] + [i for i in A_markers if i in adata_oprescu_d35.var_names], legend_loc='on data', ncols=3, cmap=magma)

In [ ]:
sc.pl.umap(adata_oprescu_d5, color=['leiden', 'Tnc', 'Tnmd', 'Pdgfra', 'Lum', 'Prg4', 'Pdpn'] + [i for i in A_markers if i in adata_oprescu_d5.var_names], legend_loc='on data', ncols=3, cmap=magma)

In [ ]:
sc.pl.umap(adata_oprescu_d10, color=['leiden', 'Tnc', 'Tnmd', 'Pdgfra', 'Lum', 'Prg4', 'Pdpn'] + [i for i in A_markers if i in adata_oprescu_d10.var_names], legend_loc='on data', ncols=3, cmap=magma)

In [ ]:
sc.pl.umap(adata_oprescu_d21, color=['leiden', 'Tnc', 'Tnmd', 'Pdgfra', 'Lum', 'Prg4', 'Pdpn'] + [i for i in A_markers if i in adata_oprescu_d21.var_names], legend_loc='on data', ncols=3, cmap=magma)

In [ ]:
dict_markers_major_populations = {
    'Kranocyte': ['Rasgrp2', 'Tenm2', 'Inpp4b', 'Foxd2os', 'Malt1', 'Gfra2', 'Shisa3', 'Malt1', 'Thrsp', 'Gpld1', 'Smim41', 'Plxdc1', 
                  'Dlk1', 'Fetub', 'Saa1', 'Gria1', 'Greb1', 'Col9a2', 'Gli1', 'Cst6'], 
    'FAP': ['Ly6a', 'Nova1', 'Fstl1', 'Ifi205', 'Slfn5', 'Fbln2', 'Dpep1', 'Fn1', 'Pdgfra', 'Lbp', 'Ifi204'],
    'Tenocyte': ['Tnmd', 'Cilp2', 'Chad', 'Scx', 'Mkx', 'Rflnb', 'Ptx4', 'Gas2', 'Kctd1', 'Edil3', 'Col11a2', 'Sox6', 'Scube2', 'Cdh2', 'Matn4'],
    'Myonuclei': ['Tnnc2', 'Myh4', 'Acta1', 'Ckm', 'Tpm2', 'Eno3', 'Slc25a4'],
    'Pericytes': ['Rgs5', 'Notch3', 'Myl9', 'Ndufa4l2', 'Itga7', 'Myh11', 'Pln', 'Abcc9'],
    'Satellite': ['Dmd', 'Pax7', 'Chodl', 'Fgfr4', 'Notch3', 'Tenm4', 'Peg3', 'Fry'], 
    'Endothelial': ['Egfl7', 'Ptprb', 'Cdh5', 'Vwf', 'Esam', 'Cd36', 'Flt1', 'Tspan13', 'Emcn'],
    'Immune': ['Lyz2', 'Ckb', 'Tyrobp', 'Iqgap1', 'Laptm5', 'Ctsc', 'Ehd4', 'Ctss', 'Fcer1g', 'Ccl4', 'C1qb',],
    'Neutrophil': ['S100a8', 'S100a9', 'Itgam', 'Cd14', ],
    'Mono/Macrophage': ['Csf1r', 'Adgre1', 'Cxcl3', 'Ccl6'],
}

In [ ]:
for adata_oprescu in [adata_oprescu_d0, adata_oprescu_d05, adata_oprescu_d2, adata_oprescu_d35, adata_oprescu_d5, adata_oprescu_d10, adata_oprescu_d21]:
    assign_cats(
    adata_oprescu,
    dict_markers_major_populations,
    column_groupby='leiden', 
    key_added='major_population',
    quantile_gene_sel=0.7,
    min_score=0.3, 
)

In [ ]:
for adata_oprescu in [adata_oprescu_d0, adata_oprescu_d05, adata_oprescu_d2, adata_oprescu_d35, adata_oprescu_d5, adata_oprescu_d10, adata_oprescu_d21]:
    sc.pl.umap(adata_oprescu, color=['major_population'], ncols=5, cmap=magma)
    sc.pl.umap(adata_oprescu, color=[i for i in A_markers if i in adata_oprescu.var_names], ncols=5, cmap=magma)
    

## Check collocalization of krano cells

In [ ]:
adata_oprescu_merges = sc.concat([adata_oprescu_d0, adata_oprescu_d05, adata_oprescu_d2, adata_oprescu_d35, adata_oprescu_d5, adata_oprescu_d10, adata_oprescu_d21], axis=0, join='outer')


sc.pp.pca(adata_oprescu_merges, random_state=seed, n_comps=30)
sce.pp.harmony_integrate(adata_oprescu_merges, 'batch', max_iter_harmony=20, random_state=seed)
sc.pp.neighbors(adata_oprescu_merges, random_state=seed, use_rep='X_pca_harmony', n_neighbors=int(len(adata_oprescu_merges) ** 0.5 // 2), metric='cosine')
sc.tl.umap(adata_oprescu_merges, min_dist=0.1, random_state=seed)

sc.tl.leiden(adata_oprescu_merges, resolution=0.3, random_state=seed, key_added='leiden_0.3')
sc.tl.leiden(adata_oprescu_merges, resolution=3, random_state=seed, key_added='leiden_3')

In [ ]:
sc.pl.umap(adata_oprescu_merges, color=['batch',], ncols=2, cmap=magma)
sc.pl.umap(adata_oprescu_merges, color=['major_population'], ncols=2, cmap=magma)
sc.pl.umap(adata_oprescu_merges, color=['leiden_0.3'], ncols=2, cmap=magma)
sc.pl.umap(adata_oprescu_merges, color=['leiden_3'], ncols=2, cmap=magma)


sc.pl.umap(adata_oprescu_merges, color=[i for i in A_markers if i in adata_oprescu.var_names], ncols=2, cmap=magma)

In [ ]:
sc.pl.umap(adata_oprescu_merges, color=['major_population', 'Pdgfra', 'Dcn', 'Col1a1'], ncols=2, cmap=magma)

In [ ]:
# Genes upregulated in cluster 1 vs 5
sc.tl.rank_genes_groups(adata_oprescu_merges, groupby='leiden_0.3', method='wilcoxon', key_added='rank_genes_leiden_0.3', groups=['1'], reference='5', use_raw=False)
sc.pl.rank_genes_groups_tracksplot(adata_oprescu_merges, key='rank_genes_leiden_0.3', n_genes=60, )

In [ ]:
# Genes upregulated in cluster 5 vs 1
sc.tl.rank_genes_groups(adata_oprescu_merges, groupby='leiden_0.3', method='wilcoxon', key_added='rank_genes_leiden_0.3', groups=['5'], reference='1', use_raw=False)
sc.pl.rank_genes_groups_tracksplot(adata_oprescu_merges, key='rank_genes_leiden_0.3', n_genes=60, )

In [ ]:
# IMPORTANT: SELECT CORRECT CLUSTER HERE!
CLUSTER_FAPS = '1'
adata_oprescu_merge_FAP = adata_oprescu_merges[adata_oprescu_merges.obs['leiden_0.3'] == CLUSTER_FAPS]

In [ ]:
sc.pp.filter_genes(adata_oprescu_merge_FAP, min_counts=1)

sc.pp.pca(adata_oprescu_merge_FAP, random_state=seed, n_comps=30)
sce.pp.harmony_integrate(adata_oprescu_merge_FAP, 'batch', max_iter_harmony=20, random_state=seed)
sc.pp.neighbors(adata_oprescu_merge_FAP, random_state=seed, use_rep='X_pca_harmony', n_neighbors=int(len(adata_oprescu_merge_FAP) ** 0.5 // 2), metric='cosine')
sc.tl.umap(adata_oprescu_merge_FAP, min_dist=0.1, random_state=seed)

In [ ]:
sc.pl.umap(adata_oprescu_merge_FAP, color=['major_population'])
sc.pl.umap(adata_oprescu_merge_FAP, color=['batch'])

fig, axs = plt.subplots(2, 4, figsize=(20, 10))
for idx, batch in enumerate(adata_oprescu_merge_FAP.obs['batch'].unique()):
    sc.pl.umap(adata_oprescu_merge_FAP, 
               ax = axs.flatten()[idx], show=False, na_color="#bcbcbc", s=5)
    sc.pl.umap(adata_oprescu_merge_FAP[adata_oprescu_merge_FAP.obs['batch'] == batch], 
               ax = axs.flatten()[idx], na_color="#252525", show=False, s=8)

# sc.pl.umap(adata_oprescu_merge_FAP, color=A_markers, cmap=magma, use_raw=False, ncols=4)

In [ ]:
dict_markers_FAP_TNC = {
    'FAP_A': ['Sema3c', 'Efemp1', 'Dpp4', 'Anxa3', 'Pi16', 'Efhd1', 'Cd248',
       'Creb5', 'Fn1', 'Fbn1', 'Limch1', 'Emilin2', 'Cd55', 'Pcolce2',
       'Gfpt2', 'Sdk1', 'Adgrd1', 'Ugdh', 'Mfap5', 'Opcml'],
    'FAP_B': ['Smoc2', 'Col4a1', 'Col15a1', 'Hmcn2', 'Hspg2', 'Lamb1', 'Hsd11b1',
       'Ccl11', 'Crispld2', 'Col4a2', 'Cxcl14', 'Grm8', 'Mme', 'Angptl1',
       'Enpp2', 'Ret', 'Col6a2', 'Col5a3', 'Abca8a', 'Lamc1'],
    'FAP_CDEF': ['Cst3', 'Gas6', 'Mgp', 'Igfbp7', 'Bgn', 'Gdf10', 'Gas1', 'Apod',
       'Cygb', 'Cfh', 'Srpx', 'Il11ra1', 'Fbln1', 'Boc', 'Nrp1', 'Nfib',
       'Olfml3', 'Penk', 'Fgl2', 'Prg4'],
              'Teno_D': ['Sparcl1', 'F2r', 'Col4a2', 'Ptpre', 'Col22a1', 'Fbn2', 'Col18a1', 'Rapgef4', 'Lrrn2', 'Reln', 'Adam23',],
    'Teno_C': ['Ccdc3', 'Bmpr1b', 'Wif1', 'Kera', 'C3', 'Sema3b', 'Sema3c', 'Chrdl1', 'Grem2', 'Itga2', 'Hpgd', 'Bmp3', 'Clu'],
    'Teno_B': ['Gpx3', 'Col8a1', 'Bicc1', 'Runx2', 'Smoc2', 'Camk4', 'Spon1', 'Wnt16', 'Slit2', 'Ostn', 'H19', 'Igf2'],
        'Krano': ['Rasgrp2', 'Tenm2', 'Inpp4b', 'Foxd2os', 'Malt1', 'Gfra2', 'Shisa3', 'Malt1', 'Thrsp', 'Gpld1', 'Smim41', 'Plxdc1', 
                  'Dlk1', 'Fetub', 'Saa1', 'Gria1', 'Greb1', 'Col9a2', 'Gli1', 'Cst6'],
    'Myonuclei': ['Tnnc2', 'Chchd10', 'Eno3', 'Tnni2', 'Tnnt3', 'Acta1', 'Myl1', 'Pgam2', 'Ckm', 'Cox6a2']
    }


for key, val in dict_markers_FAP_TNC.items():
    print(key)
    sc.pl.umap(adata_oprescu_merge_FAP, color=[i for i in val if i in adata_oprescu_merge_FAP.var_names], cmap=magma, use_raw=False, ncols=4)

In [ ]:
sc.pl.umap(adata_oprescu_merges, color=['major_population'], ncols=2, cmap=magma)

sc.pl.umap(adata_oprescu_merges, color=['Pdgfra', 'Dcn', 'Col1a1', 'Tnnc2', 'Tnnt3', 'Mylpf', 'Acta1', 'Tnni2', 'Ckm', 'Tcap', 'Cox6a2', 'Pvalb', 'Cox8b', 'Myoz1', 
                        'Atp2a1', 'Neb', 'Ak1', 'Cox7a1'], ncols=3, cmap=magma)

# Managing subpopulations

## Noninjured

In [ ]:
adata_oprescu_merge_FAP_NI = adata_oprescu_merge_FAP[adata_oprescu_merge_FAP.obs['batch'] == 'Noninjured']

preprocessing_adata_sub(adata_oprescu_merge_FAP_NI, integrate=False)
sc.tl.umap(adata_oprescu_merge_FAP_NI, min_dist=0.1, random_state=seed)
sc.tl.leiden(adata_oprescu_merge_FAP_NI, resolution=5, random_state=seed, key_added='leiden_sub')

In [ ]:
assign_cats(
    adata_oprescu_merge_FAP_NI,
    dict_markers_FAP_TNC,
    column_groupby='leiden_sub', 
    key_added='sub_population',
    quantile_gene_sel=0.55,
    min_score=0.45, 
)

In [ ]:
sc.pl.umap(adata_oprescu_merge_FAP_NI, color=['sub_population'], ncols=2, cmap=magma)
sc.pl.umap(adata_oprescu_merge_FAP_NI, color=['leiden_sub'], ncols=2, cmap=magma)

# for key, val in dict_markers_FAP_TNC.items():
#     print(key)
#     sc.pl.umap(adata_oprescu_merge_FAP_NI, color=[i for i in val if i in adata_oprescu_merge_FAP_NI.var_names], cmap=magma, use_raw=False, ncols=4)

## D0.5

In [ ]:
adata_oprescu_merge_FAP_D05 = adata_oprescu_merge_FAP[adata_oprescu_merge_FAP.obs['batch'] == 'X0.5.DPI']

preprocessing_adata_sub(adata_oprescu_merge_FAP_D05, integrate=False)
sc.tl.umap(adata_oprescu_merge_FAP_D05, min_dist=0.1, random_state=seed)
sc.tl.leiden(adata_oprescu_merge_FAP_D05, resolution=1, random_state=seed, key_added='leiden_sub')

In [ ]:
assign_cats(
    adata_oprescu_merge_FAP_D05,
    dict_markers_FAP_TNC,
    column_groupby='leiden_sub', 
    key_added='sub_population',
    quantile_gene_sel=0.55,
    min_score=0.45, 
)

In [ ]:
sc.pl.umap(adata_oprescu_merge_FAP_D05, color=['sub_population'], ncols=2, cmap=magma)
sc.pl.umap(adata_oprescu_merge_FAP_D05, color=['leiden_sub'], ncols=2, cmap=magma)

# for key, val in dict_markers_FAP_TNC.items():
#     print(key)
#     try:
#         sc.pl.umap(adata_oprescu_merge_FAP_D05, color=[i for i in val if i in adata_oprescu_merge_FAP_D05.var_names], cmap=magma, use_raw=False, ncols=4)
#     except:
#         print(f"Error plotting {key}")

## D2

In [ ]:
adata_oprescu_merge_FAP_D2 = adata_oprescu_merge_FAP[adata_oprescu_merge_FAP.obs['batch'] == 'X2.DPI']

preprocessing_adata_sub(adata_oprescu_merge_FAP_D2, integrate=False)
sc.tl.umap(adata_oprescu_merge_FAP_D2, min_dist=0.1, random_state=seed)
sc.tl.leiden(adata_oprescu_merge_FAP_D2, resolution=1, random_state=seed, key_added='leiden_sub')

In [ ]:
assign_cats(
    adata_oprescu_merge_FAP_D2,
    dict_markers_FAP_TNC,
    column_groupby='leiden_sub', 
    key_added='sub_population',
    quantile_gene_sel=0.65,
    min_score=0.55, 
)

In [ ]:
sc.pl.umap(adata_oprescu_merge_FAP_D2, color=['sub_population'], ncols=2, cmap=magma)
sc.pl.umap(adata_oprescu_merge_FAP_D2, color=['leiden_sub'], ncols=2, cmap=magma)

# for key, val in dict_markers_FAP_TNC.items():
#     print(key)
#     try:
#         sc.pl.umap(adata_oprescu_merge_FAP_D2, color=[i for i in val if i in adata_oprescu_merge_FAP_D2.var_names], cmap=magma, use_raw=False, ncols=4)
#     except:
#         print(f"Error plotting {key}")

## D3.5

In [ ]:
adata_oprescu_merge_FAP_D35 = adata_oprescu_merge_FAP[adata_oprescu_merge_FAP.obs['batch'] == 'X3.5.DPI']

preprocessing_adata_sub(adata_oprescu_merge_FAP_D35, integrate=False)
sc.tl.umap(adata_oprescu_merge_FAP_D35, min_dist=0.1, random_state=seed)
sc.tl.leiden(adata_oprescu_merge_FAP_D35, resolution=1, random_state=seed, key_added='leiden_sub')

In [ ]:
assign_cats(
    adata_oprescu_merge_FAP_D35,
    dict_markers_FAP_TNC,
    column_groupby='leiden_sub', 
    key_added='sub_population',
    quantile_gene_sel=0.65,
    min_score=0.55, 
)

In [ ]:
sc.pl.umap(adata_oprescu_merge_FAP_D35, color=['sub_population'], ncols=2, cmap=magma)
sc.pl.umap(adata_oprescu_merge_FAP_D35, color=['leiden_sub'], ncols=2, cmap=magma)

# for key, val in dict_markers_FAP_TNC.items():
#     print(key)
#     sc.pl.umap(adata_oprescu_merge_FAP_D35, color=[i for i in val if i in adata_oprescu_merge_FAP_D35.var_names], cmap=magma, use_raw=False, ncols=4)

## D5

In [ ]:
adata_oprescu_merge_FAP_D5 = adata_oprescu_merge_FAP[adata_oprescu_merge_FAP.obs['batch'] == 'X5.DPI']

preprocessing_adata_sub(adata_oprescu_merge_FAP_D5, integrate=False)
sc.tl.umap(adata_oprescu_merge_FAP_D5, min_dist=0.1, random_state=seed)
sc.tl.leiden(adata_oprescu_merge_FAP_D5, resolution=1, random_state=seed, key_added='leiden_sub')

In [ ]:
assign_cats(
    adata_oprescu_merge_FAP_D5,
    dict_markers_FAP_TNC,
    column_groupby='leiden_sub', 
    key_added='sub_population',
    quantile_gene_sel=0.95,
    min_score=0.9, 
)

In [ ]:
sc.pl.umap(adata_oprescu_merge_FAP_D5, color=['sub_population'], ncols=2, cmap=magma)
sc.pl.umap(adata_oprescu_merge_FAP_D5, color=['leiden_sub'], ncols=2, cmap=magma)

# for key, val in dict_markers_FAP_TNC.items():
#     print(key)
#     sc.pl.umap(adata_oprescu_merge_FAP_D5, color=[i for i in val if i in adata_oprescu_merge_FAP_D5.var_names], cmap=magma, use_raw=False, ncols=4)

## D10

In [ ]:
adata_oprescu_merge_FAP_D10 = adata_oprescu_merge_FAP[adata_oprescu_merge_FAP.obs['batch'] == 'X10.DPI']

preprocessing_adata_sub(adata_oprescu_merge_FAP_D10, integrate=False)
sc.tl.umap(adata_oprescu_merge_FAP_D10, min_dist=0.1, random_state=seed)
sc.tl.leiden(adata_oprescu_merge_FAP_D10, resolution=1, random_state=seed, key_added='leiden_sub')

In [ ]:
assign_cats(
    adata_oprescu_merge_FAP_D10,
    dict_markers_FAP_TNC,
    column_groupby='leiden_sub', 
    key_added='sub_population',
    quantile_gene_sel=0.8,
    min_score=0.8, 
)

In [ ]:
sc.pl.umap(adata_oprescu_merge_FAP_D10, color=['sub_population'], ncols=2, cmap=magma)
sc.pl.umap(adata_oprescu_merge_FAP_D10, color=['leiden_sub'], ncols=2, cmap=magma)

# for key, val in dict_markers_FAP_TNC.items():
#     print(key)
#     sc.pl.umap(adata_oprescu_merge_FAP_D10, color=[i for i in val if i in adata_oprescu_merge_FAP_D10.var_names], cmap=magma, use_raw=False, ncols=4)

## D21

In [ ]:
adata_oprescu_merge_FAP_D21 = adata_oprescu_merge_FAP[adata_oprescu_merge_FAP.obs['batch'] == 'X21.DPI']

preprocessing_adata_sub(adata_oprescu_merge_FAP_D21, integrate=False)
sc.tl.umap(adata_oprescu_merge_FAP_D21, min_dist=0.1, random_state=seed)
sc.tl.leiden(adata_oprescu_merge_FAP_D21, resolution=1.5, random_state=seed, key_added='leiden_sub')

In [ ]:
assign_cats(
    adata_oprescu_merge_FAP_D21,
    dict_markers_FAP_TNC,
    column_groupby='leiden_sub', 
    key_added='sub_population',
    quantile_gene_sel=0.8,
    min_score=0.5, 
)

In [ ]:
sc.pl.umap(adata_oprescu_merge_FAP_D21, color=['sub_population'], ncols=2, cmap=magma)
sc.pl.umap(adata_oprescu_merge_FAP_D21, color=['leiden_sub'], ncols=2, cmap=magma)

# for key, val in dict_markers_FAP_TNC.items():
#     print(key)
#     sc.pl.umap(adata_oprescu_merge_FAP_D21, color=[i for i in val if i in adata_oprescu_merge_FAP_D21.var_names], cmap=magma, use_raw=False, ncols=4)

# Subpopulation QC

During some analyses we observed that some specific datasets suffer a decoupling, that it, there are "two copies" with the same populations. This is very apparent, for instance, at D10.

In [ ]:
sc.tl.leiden(adata_oprescu_merge_FAP_D10, resolution=0.1, key_added='leiden_decoupling')
sc.pl.umap(adata_oprescu_merge_FAP_D10, color='leiden_decoupling')

In [ ]:
# TODO: MIRAR QUE COÑO PASA AQUI!!
sc.tl.rank_genes_groups(adata_oprescu_merge_FAP_D10, groupby='leiden_decoupling')
df_pvals_decoupling = plot_volcano(adata_oprescu_merge_FAP_D10, '0', pval_threshold=1e-20, lfc_threshold=0.5, topn=20, bottomn=5, return_df=True, plot_positive_only=False, xlim=(-3, 7))
sc.pl.umap(adata_oprescu_merge_FAP_D10, color=df_pvals_decoupling.sort_values(by='pvalxlfc', ascending=False)['gene'].values[:10], cmap=magma, use_raw=False, ncols=5)

df_pvals_decoupling.to_csv('results/oprescu/stress_DEGs.csv')

GOs of the markers show that many of them are linked to a stress response. This is very likely if the sample handling step included a harsh processing. Thus, we are going to flag and remove stressed cells as a safety measure to ensure that there are no subsequent biases.

In [ ]:
dict_markers_stress = {'stress_response': [
    'Fosl1','Fosl2','Fosb','Myc','Atf3','Ier5','Zfp36',
    'Nr4a1','Nr4a2','Nr4a3','Nfkbia','Dusp5','Dusp10',
    'Hmox1','Txnrd1','Mt1','Mt2','Hspa8','Dnajb1','Dnaja4',
    'Bag3','Pim1','Pim3','Trib1','Il6','Ptgs2','Ccl2','Ccl7',
    'Tnfaip6','Ptx3','Has1'
    ]}

In [ ]:
def flag_and_clear_stress(adata_sub_day):
    adata_sub_day = adata_sub_day.copy()

    sc.tl.leiden(adata_sub_day, resolution = 4, key_added='leiden_stress')
    assign_cats(adata_sub_day, dict_cats = dict_markers_stress, 
                column_groupby='leiden_stress', key_added='assigned_cats_stress', 
                quantile_gene_sel=0.99, min_score=0.9)
    
    stress_genes = [g for g in dict_markers_stress['stress_response'] if g in adata_sub_day.var_names]

    sc.tl.score_genes(adata_sub_day, stress_genes, score_name='stress_score')
    sc.pl.umap(adata_sub_day, color=['stress_score'])
    sc.pl.umap(adata_sub_day, color=['assigned_cats_stress'])
    sc.pl.umap(adata_sub_day, color=['sub_population'])

    sc.pl.umap(adata_sub_day, color=[i for i in dict_markers_stress['stress_response'] if i in adata_sub_day.var_names], 
    cmap=magma, ncols=5)

    return adata_sub_day


In [ ]:
adata_oprescu_merge_FAP_NI_cleared = flag_and_clear_stress(adata_oprescu_merge_FAP_NI)
adata_oprescu_merge_FAP_D05_cleared = flag_and_clear_stress(adata_oprescu_merge_FAP_D05)
adata_oprescu_merge_FAP_D2_cleared = flag_and_clear_stress(adata_oprescu_merge_FAP_D2)
adata_oprescu_merge_FAP_D35_cleared = flag_and_clear_stress(adata_oprescu_merge_FAP_D35)
adata_oprescu_merge_FAP_D5_cleared = flag_and_clear_stress(adata_oprescu_merge_FAP_D5)
adata_oprescu_merge_FAP_D10_cleared = flag_and_clear_stress(adata_oprescu_merge_FAP_D10)
adata_oprescu_merge_FAP_D21_cleared = flag_and_clear_stress(adata_oprescu_merge_FAP_D21)

## Reprocessing the clusters

In [ ]:
list_sub_names = adata_oprescu_merges.obs['batch'].cat.categories
list_sub_adatas = [adata_oprescu_merge_FAP_NI_cleared, adata_oprescu_merge_FAP_D05_cleared, adata_oprescu_merge_FAP_D2_cleared, 
                   adata_oprescu_merge_FAP_D35_cleared, adata_oprescu_merge_FAP_D5_cleared, adata_oprescu_merge_FAP_D10_cleared, adata_oprescu_merge_FAP_D21_cleared]


In [ ]:
for name, adata in zip(list_sub_names, list_sub_adatas):
    print(name)
    preprocessing_adata_sub(adata, integrate=False)
    sc.tl.umap(adata)
    sc.pl.umap(adata, color='sub_population')

In [ ]:
adata_oprescu_merge_FAP_cleared = sc.concat(list_sub_adatas)

sc.pp.filter_genes(adata_oprescu_merge_FAP_cleared, min_counts=1)

sc.pp.pca(adata_oprescu_merge_FAP_cleared, random_state=seed, n_comps=30)
sce.pp.harmony_integrate(adata_oprescu_merge_FAP_cleared, 'batch', max_iter_harmony=20, random_state=seed)
sc.pp.neighbors(adata_oprescu_merge_FAP_cleared, random_state=seed, use_rep='X_pca_harmony', n_neighbors=int(len(adata_oprescu_merge_FAP_cleared) ** 0.5 // 3), metric='cosine')
sc.tl.umap(adata_oprescu_merge_FAP_cleared, min_dist=0.05, random_state=seed)

In [ ]:
sc.pl.umap(adata_oprescu_merge_FAP_cleared, color='sub_population')

In [ ]:
adata_oprescu_merge_FAP_cleared.write_h5ad('data/processed/oprescu_processed_clean_FAP.h5ad')